In [5]:
from tennis_data_pipeline.datasources.wta import WtaApiClient

# WtaApiClient wraps the ad hoc requests calls from earlier exploration:
# no custom headers needed, handles pagination, and returns a flat DataFrame
# keyed on `tournament_group_id` (confirmed stable across years).
client = WtaApiClient()
df_tournaments = client.get_tournaments(2025)
df_tournaments

,tournament_group_id,group_name,level,title,year,start_date,end_date,surface,in_outdoor,city,country,singles_draw_size,doubles_draw_size,prize_money,singles_champion
0,2084,UNITED CUP,WTA 500,"United Cup - Australia, AUS",2025,2024-12-27,2025-01-05,Hard,O,SYDNEY + PERTH,AUS,0,0,5125000,NaN
1,800,BRISBANE,WTA 500,Brisbane International presented by Evie - Bri...,2025,2024-12-29,2025-01-05,Hard,O,BRISBANE,AUS,48,16,1520600,Aryna Sabalenka
2,2096,CANBERRA 125,WTA 125,"Workday Canberra International - Canberra, AUS",2025,2024-12-30,2025-01-04,Hard,O,CANBERRA,AUS,32,16,200000,Aoi Ito
3,1049,AUCKLAND,WTA 250,"ASB Classic - Auckland, NZL",2025,2024-12-30,2025-01-05,Hard,O,AUCKLAND,NZL,32,16,275094,Clara Tauson
4,4440,NAIROBI,ITF,"WTT W35 - NAIROBI, KENYA",2025,2024-12-30,2025-01-05,Clay,O,,KEN,32,16,30000,Joanna Garland
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,2934,SOLAPUR,ITF,"WTT W35- SOLAPUR, INDIA",2025,2025-12-22,2025-12-28,Hard,O,,IND,32,16,30000,Vaidehee Chaudhari
744,4947,MONASTIR 51,ITF,"WTT W15 - MONASTIR 51, TUNISIA",2025,2025-12-22,2025-12-28,Hard,O,,TUN,32,16,15000,Sapfo Sakellaridi
745,2742,MONASTIR,ITF,"WTT W15 - MONASTIR, TUNISIA",2026,2025-12-29,2026-01-04,Hard,O,,TUN,32,16,15000,Lan Mi
746,4440,NAIROBI,ITF,"WTT W35 - NAIROBI, KENYA",2026,2025-12-29,2026-01-04,Clay,O,,KEN,32,16,30000,Angella Okutoyi


In [6]:
print(f"{len(df_tournaments)} tournaments returned for 2025")

748 tournaments returned for 2025


In [7]:
print(df_tournaments["level"].value_counts())

# Tennis-Data UK only covers the main tour (not ITF/W15-W100 events), so this is
# likely the subset we'd actually try to match against.
main_tour = df_tournaments[~df_tournaments["level"].eq("ITF")]
main_tour[["tournament_group_id", "group_name", "level", "title", "city", "start_date", "end_date"]]

level
ITF           647
WTA 125        49
WTA 250        19
WTA 500        17
WTA 1000       10
Grand Slam      4
Finals          1
Name: count, dtype: int64


,tournament_group_id,group_name,level,title,city,start_date,end_date
0,2084,UNITED CUP,WTA 500,"United Cup - Australia, AUS",SYDNEY + PERTH,2024-12-27,2025-01-05
1,800,BRISBANE,WTA 500,Brisbane International presented by Evie - Bri...,BRISBANE,2024-12-29,2025-01-05
2,2096,CANBERRA 125,WTA 125,"Workday Canberra International - Canberra, AUS",CANBERRA,2024-12-30,2025-01-04
3,1049,AUCKLAND,WTA 250,"ASB Classic - Auckland, NZL",AUCKLAND,2024-12-30,2025-01-05
6,2014,ADELAIDE,WTA 500,"Adelaide International - Adelaide, AUS",ADELAIDE,2025-01-06,2025-01-11
...,...,...,...,...,...,...,...
694,2076,COLINA 125,WTA 125,"LP Open by IND - Colina, CHI",COLINA,2025-11-17,2025-11-23
709,2052,BUENOS AIRES 125,WTA 125,"IEB+ Argentina Open - Buenos Aires, ARG",BUENOS AIRES,2025-11-24,2025-11-30
722,1118,QUITO 125,WTA 125,"Quito Open - Quito, ECU",QUITO,2025-12-01,2025-12-07
723,2056,ANGERS 125,WTA 125,"Open Angers Loire Trélazé - Angers, FRA",ANGERS,2025-12-01,2025-12-07


In [8]:
# Check whether `tournamentGroup.id` is stable year-over-year for the same
# tournament (this is the key question - if stable, it's a much better anchor
# id than fuzzy name-matching).
df_2019 = client.get_tournaments(2019)

check_names = ["ADELAIDE", "INDIAN WELLS", "DUBAI", "DOHA", "AUCKLAND"]
compare = (
    df_tournaments[df_tournaments["group_name"].isin(check_names)][["group_name", "tournament_group_id"]]
    .drop_duplicates()
    .merge(
        df_2019[df_2019["group_name"].isin(check_names)][
            ["group_name", "tournament_group_id"]
        ].drop_duplicates(),
        on="group_name",
        suffixes=("_2025", "_2019"),
        how="outer",
    )
)
compare

,group_name,tournament_group_id_2025,tournament_group_id_2019
0,ADELAIDE,2014,NaN
1,AUCKLAND,1049,1049.0
2,DOHA,1003,1003.0
3,DUBAI,718,718.0
4,DUBAI,718,2300.0
5,DUBAI,2300,718.0
6,DUBAI,2300,2300.0
7,INDIAN WELLS,609,609.0
